# Workshop: Transformations on RetailHub Silver Data

**Learning objective:** Practice the core skills of exam domain 3 — *Data Transformation and Modeling (22%)*: multi-table joins, broadcast joins with plan verification, window-based deduplication, array explode, exact vs approximate aggregations, analytic window functions, and one Spark tuning experiment.

**Expected duration:** ~45 minutes, self-study (Task 7 is a stretch task)

> ⏭️ **SELF-STUDY LAB** — not run in class in the 6-hour course format. Work through it after `05a_transformations`; hints in `notebooks/guides/lab_06_transformations_guide.ipynb`, full answers in `notebooks/solution/lab_06_transformations_solution.ipynb`.

**Prerequisite:** Complete the `05a — Transformations & Modeling` demo before starting this lab.

| Task | Topic |
|------|-------|
| 1 | Multi-table joins — inner vs left across two keys |
| 2 | Broadcast join — verify with `df.explain()` |
| 3 | Deduplication with `row_number()` — the stream/batch overlap |
| 4 | Arrays — build with `collect_set`, unpack with `explode` |
| 5 | Aggregations — `count`, `approx_count_distinct`, `avg` + `summary()` |
| 6 | Window functions — `rank` and `lag` revenue per customer segment |
| 7 | Challenge — tuning experiment with `spark.sql.shuffle.partitions` |

> **SOLUTION NOTEBOOK** — full answers for `lab_06_transformations`. Runs top-to-bottom on a fresh participant catalog.

## Setup

The preparation cell below builds three **Silver** tables from the RetailHub raw files — typed, renamed, ready for modeling. It is provided; just run it.

In [ ]:
%run ../setup/00_setup

### Configuration

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
import io, contextlib, time

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")

In [ ]:
# -- Provided: build Silver tables from raw RetailHub files (no TODOs here) --
ORDERS_JSON      = f"{DATASET_PATH}/orders/orders_batch.json"
STREAM_001_JSON  = f"{DATASET_PATH}/orders/stream/orders_stream_001.json"
CUSTOMERS_CSV    = f"{DATASET_PATH}/customers/customers.csv"
PRODUCTS_PARQUET = f"{DATASET_PATH}/products/products.parquet"

ORDERS_SILVER    = f"{CATALOG}.{SILVER_SCHEMA}.orders_silver_lab06"
CUSTOMERS_SILVER = f"{CATALOG}.{SILVER_SCHEMA}.customers_silver_lab06"
PRODUCTS_SILVER  = f"{CATALOG}.{SILVER_SCHEMA}.products_silver_lab06"

(spark.read.format("json").load(ORDERS_JSON)
    .withColumn("order_ts", F.col("order_datetime").cast("timestamp"))
    .withColumn("order_month", F.date_trunc("month", F.col("order_ts")).cast("date"))
    .write.mode("overwrite").saveAsTable(ORDERS_SILVER))

(spark.read.format("csv").option("header", True).option("inferSchema", True).load(CUSTOMERS_CSV)
    .write.mode("overwrite").saveAsTable(CUSTOMERS_SILVER))

(spark.read.format("parquet").load(PRODUCTS_PARQUET)
    .write.mode("overwrite").saveAsTable(PRODUCTS_SILVER))

orders_df    = spark.table(ORDERS_SILVER)
customers_df = spark.table(CUSTOMERS_SILVER)
products_df  = spark.table(PRODUCTS_SILVER)

print(f"orders    : {orders_df.count():,} rows")
print(f"customers : {customers_df.count():,} rows")
print(f"products  : {products_df.count():,} rows")

# Note: ~3% of order rows are intentionally dirty (null keys, nulls in
# order_datetime, unknown customer/product ids) — that is part of the lab.

## Task 1: Multi-Table Joins — Inner vs Left

Join orders to customers (on `customer_id`) and to products (on `product_id`) — the two join keys of the RetailHub model.

**What you need to do:**
1. Build `enriched_inner` — orders **inner**-joined to customers, then **inner**-joined to products
2. Build `enriched_left` — the same chain with **left** joins
3. Select `order_id`, `customer_id`, `customer_segment`, `product_id`, `product_name`, `total_amount` in both
4. Compare the row counts — why do they differ?

**Guidance — Task 01**

**Join syntax (DataFrame API)**
```python
df_joined = (df_left
    .join(df_right, on="key_column", how="inner")   # or how="left"
    .join(df_third, on="other_key",  how="inner"))
```
Passing the key as a string (`on="customer_id"`) automatically de-duplicates the join column in the output. A chain of two joins uses a **different key at each step** — `customer_id` first, `product_id` second.

**Inner vs left**
- `inner` keeps only rows with a match on **both** sides — orders with a null or unknown `customer_id`/`product_id` disappear.
- `left` keeps **every** order; unmatched customer/product columns become `NULL`.

**Things to think about**
- The RetailHub feed contains ~3% dirty rows. Which join type would silently drop them?
- In a medallion architecture, at which layer would you resolve unmatched keys instead of dropping them?

In [ ]:
cols = ["order_id", "customer_id", "customer_segment", "product_id", "product_name", "total_amount"]

enriched_inner = (
    orders_df
    .join(customers_df, on="customer_id", how="inner")
    .join(products_df,  on="product_id",  how="inner")
    .select(*cols)
)

enriched_left = (
    orders_df
    .join(customers_df, on="customer_id", how="left")
    .join(products_df,  on="product_id",  how="left")
    .select(*cols)
)

print(f"orders          : {orders_df.count():,}")
print(f"enriched_inner  : {enriched_inner.count():,}")
print(f"enriched_left   : {enriched_left.count():,}")
display(enriched_inner.limit(5))

In [ ]:
# -- Validation --
orders_count = orders_df.count()
inner_count  = enriched_inner.count()
left_count   = enriched_left.count()
assert set(["customer_segment", "product_name"]).issubset(enriched_inner.columns), \
    "Join output must include customer_segment and product_name"
assert left_count == orders_count, \
    f"LEFT join must keep every order: {left_count} != {orders_count}"
assert inner_count < left_count, \
    "INNER join should drop orders with unmatched/null keys (the ~3% dirty rows)"
print(f"Task 1 OK: inner={inner_count:,}, left={left_count:,} — "
      f"{left_count - inner_count:,} orders lost by the inner join")

## Task 2: Broadcast Join — Verify via the Physical Plan

Products (2k rows) is tiny next to orders (100k rows). Force a **broadcast join** and prove from the physical plan that Spark actually broadcast the small side.

**What you need to do:**
1. Join `orders_df` to `broadcast(products_df)` on `product_id`
2. Capture the output of `df.explain()` into the string `plan_str`
3. Confirm the plan contains a `Broadcast` exchange

**Guidance — Task 02**

**Broadcast hint**
```python
from pyspark.sql.functions import broadcast
df = large_df.join(broadcast(small_df), on="key")
```
`broadcast()` ships the entire small table to every executor, so the large side is **never shuffled**. Spark does this automatically for tables under `spark.sql.autoBroadcastJoinThreshold` (10 MB default) — the hint forces it regardless of statistics.

**Capturing the plan as a string**
`df.explain()` prints to stdout and returns `None` — redirect stdout to capture it (this also works on serverless / Spark Connect):
```python
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    df.explain()
plan_str = buf.getvalue()
```
Look for `BroadcastHashJoin` / `BroadcastExchange` in the output.

**Things to think about**
- Why would broadcasting the *orders* side be a bad idea?
- What happens if you broadcast a table that does not fit in executor memory?

In [ ]:
from pyspark.sql.functions import broadcast

broadcast_join_df = (
    orders_df.join(broadcast(products_df), on="product_id", how="inner")
)

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    broadcast_join_df.explain()
plan_str = buf.getvalue()

print(plan_str)

In [ ]:
# -- Validation --
assert broadcast_join_df.count() > 0, "Broadcast join should return rows"
assert "Broadcast" in plan_str, \
    "Physical plan should contain a Broadcast exchange (BroadcastHashJoin / BroadcastExchange)"
print("Task 2 OK: plan contains a Broadcast join — small side shipped to executors, no shuffle of orders")

## Task 3: Deduplication with a Window — the Stream/Batch Overlap

RetailHub's streaming feed **replays part of the batch extract**: the file `orders_stream_001.json` contains the same orders as the head of `orders_batch.json`. Union the two sources and the same `order_id` appears twice — a classic real-world ingestion overlap.

**What you need to do:**
1. Read `STREAM_001_JSON`, tag each source with a `source` column (`batch` / `stream`), union them
2. Count how many `order_id`s are duplicated
3. Deduplicate keeping the **batch** version: `row_number()` over a window partitioned by `order_id`, ordered by `source`, keep `rn = 1`

**Guidance — Task 03**

**Why not `dropDuplicates(["order_id"])`?**
It works, but you cannot control *which* duplicate survives. The window pattern makes the choice explicit — that is what interviewers (and the exam) look for:

```python
w = Window.partitionBy("order_id").orderBy("source")   # 'batch' < 'stream' alphabetically
deduped = (combined
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn"))
```

**Union pattern**
```python
a = df1.withColumn("source", F.lit("batch"))
b = df2.withColumn("source", F.lit("stream"))
combined = a.unionByName(b)
```
`unionByName` aligns columns by name instead of position — safer than `union`.

**Things to think about**
- Why does ordering by `source` guarantee the batch row wins?
- In production, what column would you order by instead (hint: ingestion timestamp)?

In [ ]:
batch_tagged  = orders_df.withColumn("source", F.lit("batch"))
stream_tagged = (
    spark.read.format("json").load(STREAM_001_JSON)
    .withColumn("order_ts", F.col("order_datetime").cast("timestamp"))
    .withColumn("order_month", F.date_trunc("month", F.col("order_ts")).cast("date"))
    .withColumn("source", F.lit("stream"))
)
combined = batch_tagged.unionByName(stream_tagged, allowMissingColumns=True)

total_rows    = combined.count()
distinct_ids  = combined.select("order_id").distinct().count()
print(f"Combined rows      : {total_rows:,}")
print(f"Distinct order_ids : {distinct_ids:,}")
print(f"Duplicated rows    : {total_rows - distinct_ids:,}   <-- stream_001 replays the batch head!")

w = Window.partitionBy("order_id").orderBy("source")
deduped = (
    combined
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)
print(f"After dedup        : {deduped.count():,}")

In [ ]:
# -- Validation --
dedup_count = deduped.count()
assert total_rows > distinct_ids, \
    "The union should contain duplicates (orders_stream_001 replays the batch head)"
assert dedup_count == distinct_ids, \
    f"Dedup must keep exactly one row per order_id: {dedup_count} != {distinct_ids}"
kept_batch = deduped.filter("source = 'batch'").count()
assert kept_batch == orders_df.count(), \
    "Every surviving duplicate should be the batch version (orderBy('source'))"
print(f"Task 3 OK: {total_rows - dedup_count:,} duplicate rows removed, batch versions kept")

## Task 4: Arrays — collect_set + explode

Orders are one-row-per-product. Build a per-customer **basket** (an array of the product ids they bought), then unpack it back to rows with `explode()`.

**What you need to do:**
1. `baskets`: group orders by `customer_id`, aggregate `collect_set("product_id")` as `products`
2. Add `basket_size` using `F.size()`
3. `exploded`: one row per `(customer_id, product_id)` using `F.explode()`

**Guidance — Task 04**

**Building an array column**
```python
baskets = (orders_df
    .groupBy("customer_id")
    .agg(F.collect_set("product_id").alias("products"))   # array of DISTINCT values
    .withColumn("basket_size", F.size("products")))
```
`collect_set` deduplicates; `collect_list` keeps duplicates. (You can also create arrays with `F.split(string_col, " ")` — same array type, different source.)

**Unpacking an array**
```python
exploded = baskets.select("customer_id", F.explode("products").alias("product_id"))
```
`explode` produces **one output row per array element**. `posexplode` also returns the element's position.

**Things to think about**
- `exploded` should have exactly as many rows as there are distinct `(customer_id, product_id)` pairs — why?
- When is it better to keep the array and use higher-order functions (`TRANSFORM`, `FILTER`) instead of exploding?

In [ ]:
baskets = (
    orders_df
    .groupBy("customer_id")
    .agg(F.collect_set("product_id").alias("products"))
    .withColumn("basket_size", F.size("products"))
)

baskets.printSchema()
display(baskets.orderBy(F.desc("basket_size")).limit(5))

exploded = baskets.select("customer_id", F.explode("products").alias("product_id"))
display(exploded.limit(5))

In [ ]:
# -- Validation --
products_dtype = dict(baskets.dtypes)["products"]
assert products_dtype.startswith("array"), f"'products' must be an array column, got {products_dtype}"
expected_pairs = orders_df.select("customer_id", "product_id").distinct().count()
exploded_count = exploded.count()
assert exploded_count == expected_pairs, \
    f"explode() should yield one row per distinct (customer, product) pair: {exploded_count} != {expected_pairs}"
print(f"Task 4 OK: {baskets.count():,} baskets -> {exploded_count:,} exploded rows")

## Task 5: Aggregations — Exact, Approximate, Descriptive

Profile revenue per customer segment with `count`, `approx_count_distinct`, and `avg`; then get full descriptive statistics of `total_amount` with `summary()`.

**What you need to do:**
1. `segment_stats`: from `enriched_left` (Task 1), group by `customer_segment` and compute `order_count`, `approx_customers` (`approx_count_distinct("customer_id")`), `avg_amount`
2. Compare `approx_count_distinct` vs exact `count_distinct` on the full orders table
3. `summary_df`: `orders_df.select("total_amount").summary()`

**Guidance — Task 05**

**Aggregation pattern**
```python
stats = (df.groupBy("segment_col").agg(
    F.count("*").alias("order_count"),
    F.approx_count_distinct("customer_id").alias("approx_customers"),
    F.round(F.avg("total_amount"), 2).alias("avg_amount"),
))
```

**Exact vs approximate**
`F.count_distinct(col)` shuffles every distinct value — exact but expensive.
`F.approx_count_distinct(col)` uses HyperLogLog++ — single pass, default error ±5%.

**summary()**
`df.summary()` returns a small DataFrame with rows `count, mean, stddev, min, 25%, 50%, 75%, max` — its first column is named `summary`.

**Things to think about**
- How large is the approximation error here, in %?
- Why does the exam emphasize `approx_count_distinct` for very large tables?

In [ ]:
segment_stats = (
    enriched_left
    .groupBy("customer_segment")
    .agg(
        F.count("*").alias("order_count"),
        F.approx_count_distinct("customer_id").alias("approx_customers"),
        F.round(F.avg("total_amount"), 2).alias("avg_amount"),
    )
)
display(segment_stats)

exact_customers  = orders_df.select(F.count_distinct("customer_id").alias("n")).first()["n"]
approx_customers = orders_df.select(F.approx_count_distinct("customer_id").alias("n")).first()["n"]
print(f"exact={exact_customers:,}  approx={approx_customers:,}  "
      f"error={abs(approx_customers - exact_customers) / exact_customers * 100:.2f}%")

summary_df = orders_df.select("total_amount").summary()
display(summary_df)

In [ ]:
# -- Validation --
assert set(["order_count", "approx_customers", "avg_amount"]).issubset(segment_stats.columns), \
    f"segment_stats missing expected columns: {segment_stats.columns}"
assert segment_stats.count() >= 2, "Expected at least 2 customer segments"
assert abs(approx_customers - exact_customers) / exact_customers < 0.10, \
    "approx_count_distinct should be within 10% of the exact count"
summary_rows = [r["summary"] for r in summary_df.collect()]
assert "mean" in summary_rows and "50%" in summary_rows, \
    f"summary() should include 'mean' and percentiles, got {summary_rows}"
print(f"Task 5 OK: approx error "
      f"{abs(approx_customers - exact_customers) / exact_customers * 100:.2f}% — summary() verified")

## Task 6: Window Functions — rank & lag Revenue per Segment

Build a monthly revenue series per customer segment, then answer two classic analytics questions with window functions:
- **rank**: which months were the best for each segment?
- **lag**: how did revenue change month-over-month?

**What you need to do:**
1. `monthly_rev`: from `enriched_left`, group by `customer_segment` + `order_month`, sum `total_amount` as `revenue` (filter out null segment/month first)
2. `revenue_rank`: `F.rank()` over a window partitioned by segment, ordered by `revenue` descending
3. `prev_revenue` + `mom_change`: `F.lag("revenue")` over a window partitioned by segment, ordered by `order_month`

**Guidance — Task 06**

**Two different windows for two different questions**
```python
w_rank = Window.partitionBy("customer_segment").orderBy(F.desc("revenue"))
w_time = Window.partitionBy("customer_segment").orderBy("order_month")

result = (monthly_rev
    .withColumn("revenue_rank", F.rank().over(w_rank))
    .withColumn("prev_revenue", F.lag("revenue", 1).over(w_time))
    .withColumn("mom_change",   F.round(F.col("revenue") - F.col("prev_revenue"), 2)))
```

| Function | Question it answers |
|---|---|
| `rank().over(orderBy revenue desc)` | "Was this the segment's #1 month?" |
| `lag(revenue, 1).over(orderBy month)` | "How much more than last month?" |

`lag` returns `NULL` for the first month of each partition — there is no previous row.

**Things to think about**
- Why must the two windows have different `orderBy` clauses?
- What would `dense_rank` return differently if two months tied?

In [ ]:
monthly_rev = (
    orders_df
    .join(customers_df, on="customer_id", how="left")
    .filter("customer_segment IS NOT NULL AND order_month IS NOT NULL")
    .groupBy("customer_segment", "order_month")
    .agg(F.round(F.sum("total_amount"), 2).alias("revenue"))
)

w_rank = Window.partitionBy("customer_segment").orderBy(F.desc("revenue"))
w_time = Window.partitionBy("customer_segment").orderBy("order_month")

windowed = (
    monthly_rev
    .withColumn("revenue_rank", F.rank().over(w_rank))
    .withColumn("prev_revenue", F.lag("revenue", 1).over(w_time))
    .withColumn("mom_change",   F.round(F.col("revenue") - F.col("prev_revenue"), 2))
)

display(windowed.orderBy("customer_segment", "order_month"))

In [ ]:
# -- Validation --
assert set(["revenue_rank", "prev_revenue", "mom_change"]).issubset(windowed.columns), \
    f"Missing window columns: {windowed.columns}"
n_segments = windowed.select("customer_segment").distinct().count()
top_months = windowed.filter("revenue_rank = 1").count()
assert top_months >= n_segments, "Each segment must have a rank-1 month"
first_months = windowed.filter("prev_revenue IS NULL").count()
assert first_months == n_segments, \
    f"lag() must be NULL exactly once per segment (first month): {first_months} != {n_segments}"
print(f"Task 6 OK: {n_segments} segments ranked, lag verified on first months")

## Task 7 (Challenge): One Tuning Experiment — `spark.sql.shuffle.partitions`

> 🏃 **Stretch** — optional in class (6-hour day): do it if you finish early, or after the course.

The demo showed the 4 exam-named tuning configs. Run the experiment yourself: time the same shuffle-heavy aggregation with the default 200 shuffle partitions and with 8, then restore the original value.

**What you need to do:**
1. Save the current value of `spark.sql.shuffle.partitions`
2. Time `orders_df.groupBy("customer_id").agg(F.sum("total_amount"))` with the config at `200` -> `t_default`
3. Time the same aggregation with the config at `8` -> `t_tuned`
4. **Restore** the original value (always clean up config changes!)

**Guidance — Task 07**

**Reading / setting session configs**
```python
original = spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.shuffle.partitions", "8")
...
spark.conf.set("spark.sql.shuffle.partitions", original)   # restore!
```

**Timing an action**
Transformations are lazy — wrap the timer around an **action**:
```python
t0 = time.time()
df.groupBy("customer_id").agg(F.sum("total_amount")).count()
elapsed = time.time() - t0
```

**Interpreting the result**
100k rows shuffled into 200 partitions = 200 tiny tasks, mostly scheduling overhead. With AQE (on by default) Spark coalesces small post-shuffle partitions, so the difference may be modest — the *pattern* (fewer partitions for small data) is what the exam tests.

**Things to think about**
- Which of the 4 named configs could you NOT change on serverless, and why?
- When would you *raise* `spark.sql.shuffle.partitions` instead?

In [ ]:
def timed_agg():
    t0 = time.time()
    orders_df.groupBy("customer_id").agg(F.sum("total_amount")).count()
    return time.time() - t0

original_shuffle = spark.conf.get("spark.sql.shuffle.partitions")

spark.conf.set("spark.sql.shuffle.partitions", "200")
t_default = timed_agg()

spark.conf.set("spark.sql.shuffle.partitions", "8")
t_tuned = timed_agg()

spark.conf.set("spark.sql.shuffle.partitions", original_shuffle)

print(f"200 partitions : {t_default:.2f}s")
print(f"  8 partitions : {t_tuned:.2f}s")
print(f"restored to    : {spark.conf.get('spark.sql.shuffle.partitions')}")

In [ ]:
# -- Validation --
assert t_default > 0 and t_tuned > 0, "Both timings must be recorded"
assert spark.conf.get("spark.sql.shuffle.partitions") == original_shuffle, \
    "spark.sql.shuffle.partitions must be restored to its original value"
print(f"Task 7 OK: default={t_default:.2f}s, tuned={t_tuned:.2f}s, config restored "
      f"(on small data the win comes from less task-scheduling overhead)")

## Summary

| Task | Topic | Key Point |
|------|-------|-----------|
| 1 | Joins | Chain joins over different keys; `left` keeps all rows, `inner` silently drops dirty ~3% |
| 2 | Broadcast join | `broadcast()` hint avoids shuffling the large side — verify with `explain()` |
| 3 | Dedup | `row_number()` over a window = *controlled* dedup; stream_001 replays the batch head |
| 4 | Arrays | `collect_set` builds arrays, `explode` unpacks — one row per element |
| 5 | Aggregations | `approx_count_distinct` (HLL++, ±5%) vs exact; `summary()` for stats |
| 6 | Windows | Different questions need different windows: `rank` by value, `lag` by time |
| 7 | Tuning | `spark.sql.shuffle.partitions` — match partition count to data size; restore configs |

> **Exam Tip:** Domain 3 (Transformation and Modeling, 22%) is the largest. Know the join types, the broadcast threshold config, window-function dedup, and `approx_count_distinct` — all appear in scenario questions.

## Cleanup

In [ ]:
for t in [ORDERS_SILVER, CUSTOMERS_SILVER, PRODUCTS_SILVER]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("Lab cleanup complete")

← [05a — Transformations & Modeling](../day2/demo/05a_transformations.ipynb) | **[ README](../../README.md)** | [Lab 06](../day2/lab/lab_06_transformations.ipynb)